# تجربة الصوت التجريبية — حكاية سميرة موسى (010)
تشغيل مجاني عبر Google Colab باستخدام VoiceTut-TTS باللهجة المصرية. لا تُرفع أي ملفات صوتية إلى GitHub تلقائيًا.

**مهم:** صوت Sarah أنثوي وليس مضمونًا أنه صوت طفلة حقيقي. هذه تجربة لتقييم أقرب صوت نسائي شاب، وصوت Omar للراوي. أصوات العرض المدمجة تحتاج مراجعة حقوقها قبل أي استخدام منشور أو تجاري.

## 1) تفعيل GPU
من قائمة Runtime اختر Change runtime type ثم T4 GPU إن كان متاحًا. التشغيل المجاني وتوفر GPU يخضعان لسياسة Google Colab.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('تنبيه: اختر GPU من إعدادات Colab قبل تحميل النموذج؛ التشغيل على CPU قد يكون بطيئًا جدًا.')

In [ ]:
!pip -q install git+https://github.com/k2-fsa/OmniVoice.git voicetut-tts soundfile

In [ ]:
from voicetut_tts import VoiceTutTTS
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
from pathlib import Path

tts = VoiceTutTTS.from_pretrained('mohammedaly22/VoiceTut-TTS')
print('تم تحميل نموذج VoiceTut-TTS')

## 2) نص الحوار المختصر
هذا مقتطف من الحكاية الأصلية دون تعديل في نص الحكاية داخل الموقع.

In [ ]:
dialogue = [
    ('narrator', 'في تلاتة مارس سنة ألف وتسعمية وسبعتاشر، اتولدت سميرة موسى في قرية سنبو الكبرى، في محافظة الغربية.'),
    ('karma', 'إيه اللي كان مميز في حلم سميرة موسى للعلم؟'),
    ('narrator', 'كانت مؤمنة إن المعرفة العلمية ما ينفعش تبقى حكر على ناس قليلة، لكن ممكن تساعد المجتمع والإنسان.'),
]

# صوت نسائي للتجربة وصوت رجالي للراوي؛ يمكن تغيير الاسمين بعد سماع عينات الأصوات.
speakers = {'karma': 'Sarah', 'narrator': 'Omar'}
outputs = []
for i, (role, text) in enumerate(dialogue, start=1):
    path = f'/content/sameera_{i:02d}_{role}.wav'
    tts.synthesize(text, speaker=speakers[role], output=path)
    audio, sr = sf.read(path)
    outputs.append((role, audio, sr, path))
    print(i, role, speakers[role], 'duration:', round(len(audio)/sr, 2), 'sec')

In [ ]:
# استمع لكل دور منفصلًا
for role, audio, sr, path in outputs:
    print(role, path)
    display(Audio(audio, rate=sr))

In [ ]:
# دمج الحوار بالتبادل مع فواصل قصيرة في ملف واحد
parts = []
for role, audio, sr, path in outputs:
    if sr != 24000:
        raise ValueError(f'Unexpected sample rate: {sr}')
    parts.extend([audio.astype(np.float32), np.zeros(int(sr * 0.35), dtype=np.float32)])
mixed = np.concatenate(parts)
final_path = '/content/sameera_010_voice_pilot.wav'
sf.write(final_path, mixed, 24000)
print('الملف التجريبي جاهز:', final_path)
display(Audio(mixed, rate=24000))

In [ ]:
from google.colab import files
files.download('/content/sameera_010_voice_pilot.wav')